In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile
import os

ZIP_PATH = "/content/drive/My Drive/AIO_Homework/RCNN/data/GARBAGE CLASSIFICATION.zip"
EXTRACT_TO = "/content/data/"

os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_TO)

print("Giải nén xong!")
print(os.listdir(EXTRACT_TO))



Giải nén xong!
['GARBAGE CLASSIFICATION']


In [3]:
DATA_ROOT = "/content/data/GARBAGE CLASSIFICATION"
# Kiểm tra
print(os.listdir(DATA_ROOT))

['data.yaml', 'test', 'valid', 'train']


In [6]:
!pip install selectivesearch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import selectivesearch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
CLASS_NAMES = ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
NUM_CLASSES = len(CLASS_NAMES)

  Preparing metadata (setup.py) ... done
  Created wheel for selectivesearch: filename=selectivesearch-0.4-py3-none-any.whl size=4336 sha256=08896b8b857a5c329d6a02b2991e9b80816dc183bf3cb1169ef1fa8de6af825d
  Stored in directory: /root/.cache/pip/wheels/7f/9b/c7/58b71f1e9fe4aa0ef8affd1c673f8818bc22a5091ea8cbbe93
Successfully built selectivesearch


In [7]:
class RoIPooling(nn.Module):

    def __init__(self, output_size=7):
        super(RoIPooling, self).__init__()
        self.output_size = output_size

    def forward(self, feature_map, rois, img_size):

        img_H, img_W = img_size
        _, C, Hf, Wf = feature_map.shape

        # Tỉ lệ scale từ ảnh gốc → feature map
        scale_h = Hf / img_H
        scale_w = Wf / img_W

        pooled_list = []

        for roi in rois:
            x1, y1, x2, y2 = roi

            fx1 = int(x1 * scale_w)
            fy1 = int(y1 * scale_h)
            fx2 = int(x2 * scale_w)
            fy2 = int(y2 * scale_h)

            fx1 = max(0, min(fx1, Wf - 1))
            fy1 = max(0, min(fy1, Hf - 1))
            fx2 = max(fx1 + 1, min(fx2, Wf))
            fy2 = max(fy1 + 1, min(fy2, Hf))

            roi_feat = feature_map[:, :, fy1:fy2, fx1:fx2]  # (1, C, roi_h, roi_w)


            pooled = F.adaptive_max_pool2d(roi_feat, self.output_size)  # (1, C, 7, 7)

            pooled_list.append(pooled.squeeze(0))  # (C, 7, 7)

        return torch.stack(pooled_list, dim=0)  # (N_rois, C, 7, 7)


In [8]:
class FastRCNN(nn.Module):

    def __init__(self, num_classes, roi_size=7):
        super(FastRCNN, self).__init__()
        self.num_classes = num_classes
        self.roi_size    = roi_size

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        self.roi_pool = RoIPooling(output_size=roi_size)

        flatten_dim = 512 * roi_size * roi_size  # = 25088

        self.shared_fc = nn.Sequential(
            nn.Linear(flatten_dim, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )

        self.cls_head = nn.Linear(4096, num_classes + 1)


        self.bbox_head = nn.Linear(4096, 4 * (num_classes + 1))

    def extract_features(self, img_tensor):

        with torch.no_grad():
            feature_map = self.backbone(img_tensor)  # (1, 512, H/32, W/32)
        return feature_map

    def forward(self, feature_map, rois, img_size):

        pooled = self.roi_pool(feature_map, rois, img_size)

        # Flatten → (N, 25088)
        pooled = pooled.view(pooled.size(0), -1)

        # Shared FC → (N, 4096)
        shared = self.shared_fc(pooled)

        # Two heads
        cls_scores  = self.cls_head(shared)   # (N, num_classes+1)
        bbox_deltas = self.bbox_head(shared)  # (N, 4*(num_classes+1))

        return cls_scores, bbox_deltas


In [10]:
def decode_bbox(proposals, bbox_deltas, cls_ids):
    d_boxes: (N, 4) [x1, y1, x2, y2]

    # Chuyển proposals sang (cx, cy, w, h)
    Px = (proposals[:, 0] + proposals[:, 2]) / 2
    Py = (proposals[:, 1] + proposals[:, 3]) / 2
    Pw = proposals[:, 2] - proposals[:, 0]
    Ph = proposals[:, 3] - proposals[:, 1]

    decoded = []
    for i in range(len(proposals)):
        cls_id = cls_ids[i].item()
        offset = cls_id * 4
        tx = bbox_deltas[i, offset + 0]
        ty = bbox_deltas[i, offset + 1]
        tw = bbox_deltas[i, offset + 2]
        th = bbox_deltas[i, offset + 3]

        Gx = tx * Pw[i] + Px[i]
        Gy = ty * Ph[i] + Py[i]
        Gw = torch.exp(tw) * Pw[i]
        Gh = torch.exp(th) * Ph[i]

        x1 = Gx - Gw / 2
        y1 = Gy - Gh / 2
        x2 = Gx + Gw / 2
        y2 = Gy + Gh / 2
        decoded.append([x1.item(), y1.item(), x2.item(), y2.item()])

    return torch.tensor(decoded)


In [11]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    model = FastRCNN(num_classes=NUM_CLASSES).to(device)
    print(model)

    # ── Giả lập 1 ảnh 600×800 ──
    dummy_img  = torch.randn(1, 3, 600, 800).to(device)
    dummy_rois = [
        [50, 30, 200, 180],   # RoI 1
        [300, 100, 500, 400], # RoI 2
        [10, 400, 150, 580],  # RoI 3
    ]
    img_size = (600, 800)

    feature_map = model.extract_features(dummy_img)
    print(f"\nFeature map shape: {feature_map.shape}")

    cls_scores, bbox_deltas = model(feature_map, dummy_rois, img_size)
    print(f"cls_scores shape:  {cls_scores.shape}")   # (3, 7)
    print(f"bbox_deltas shape: {bbox_deltas.shape}")  # (3, 28)

    total = sum(p.numel() for p in model.parameters())
    print(f"\nTotal params: {total:,}")


Device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 23.7MB/s]


FastRCNN(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tru

In [12]:
from tqdm import tqdm
import os, glob

# Dùng lại GarbageRCNNDataset từ bài 1 (với transform chuẩn)
# Chỉ cần thêm: trả về cả proposals gốc (x1,y1,x2,y2) để dùng cho bbox loss

def encode_bbox(proposals, gt_boxes):

    Px = (proposals[:, 0] + proposals[:, 2]) / 2
    Py = (proposals[:, 1] + proposals[:, 3]) / 2
    Pw = proposals[:, 2] - proposals[:, 0] + 1e-6
    Ph = proposals[:, 3] - proposals[:, 1] + 1e-6

    Gx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
    Gy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
    Gw = gt_boxes[:, 2] - gt_boxes[:, 0] + 1e-6
    Gh = gt_boxes[:, 3] - gt_boxes[:, 1] + 1e-6

    tx = (Gx - Px) / Pw
    ty = (Gy - Py) / Ph
    tw = torch.log(Gw / Pw)
    th = torch.log(Gh / Ph)

    return torch.stack([tx, ty, tw, th], dim=1)  # (N, 4)


transform = transforms.Compose([
    transforms.Resize((600, 800)),   # Fast RCNN dùng ảnh lớn hơn
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


def train_fast_rcnn(model, data_root, num_epochs=3, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    cls_criterion  = nn.CrossEntropyLoss()
    bbox_criterion = nn.SmoothL1Loss()   # Smooth L1 — đúng theo paper Fast RCNN

    img_paths = sorted(glob.glob(os.path.join(data_root, "train/images/*.jpg")))
    label_dir = os.path.join(data_root, "train/labels")

    print(f"Training Fast RCNN trên {len(img_paths)} ảnh | device={device}")

    for epoch in range(num_epochs):
        total_cls_loss, total_bbox_loss = 0, 0
        total_correct, total_samples   = 0, 0

        pbar = tqdm(img_paths, desc=f"Epoch {epoch+1}/{num_epochs}", unit="img")

        for img_path in pbar:
            img_pil = Image.open(img_path).convert("RGB")
            img_w, img_h = img_pil.size

            # Load GT boxes
            stem = os.path.splitext(os.path.basename(img_path))[0]
            label_path = os.path.join(label_dir, stem + ".txt")
            gt_boxes, gt_labels = [], []
            if os.path.exists(label_path):
                with open(label_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 5: continue
                        cls_id = int(parts[0]) + 1  # +1 cho background
                        cx, cy, w, h = map(float, parts[1:5])
                        x1 = max(0, int((cx-w/2)*img_w))
                        y1 = max(0, int((cy-h/2)*img_h))
                        x2 = min(img_w, int((cx+w/2)*img_w))
                        y2 = min(img_h, int((cy+h/2)*img_h))
                        gt_boxes.append([x1,y1,x2,y2])
                        gt_labels.append(cls_id)

            if not gt_boxes:
                continue

            # Selective Search → proposals
            img_np    = np.array(img_pil)
            _, regions = selectivesearch.selective_search(img_np, scale=500, min_size=20)
            proposals  = list({(r['rect']) for r in regions
                               if r['rect'][2] > 20 and r['rect'][3] > 20})[:100]
            rois = [[x, y, x+w, y+h] for (x,y,w,h) in proposals]
            if not rois:
                continue

            # Gán nhãn cho từng RoI (IoU matching)
            roi_labels, roi_gt_boxes = [], []
            for roi in rois:
                best_iou, best_label, best_gt = 0, 0, roi
                for gt_box, gt_label in zip(gt_boxes, gt_labels):
                    xi1 = max(roi[0], gt_box[0]); yi1 = max(roi[1], gt_box[1])
                    xi2 = min(roi[2], gt_box[2]); yi2 = min(roi[3], gt_box[3])
                    inter = max(0,xi2-xi1)*max(0,yi2-yi1)
                    area1 = (roi[2]-roi[0])*(roi[3]-roi[1])
                    area2 = (gt_box[2]-gt_box[0])*(gt_box[3]-gt_box[1])
                    iou = inter/(area1+area2-inter+1e-6)
                    if iou > best_iou:
                        best_iou, best_label, best_gt = iou, gt_label, gt_box

                if best_iou >= 0.5:
                    roi_labels.append(best_label)
                    roi_gt_boxes.append(best_gt)
                elif best_iou <= 0.3:
                    roi_labels.append(0)
                    roi_gt_boxes.append(roi)
            if not roi_labels:
                continue

            # Forward
            img_tensor  = transform(img_pil).unsqueeze(0).to(device)
            img_size    = (img_pil.height, img_pil.width)
            feature_map = model.extract_features(img_tensor)

            valid_rois = rois[:len(roi_labels)]
            cls_scores, bbox_deltas = model(feature_map, valid_rois, img_size)

            labels_t  = torch.tensor(roi_labels).to(device)
            gt_boxes_t = torch.tensor(roi_gt_boxes, dtype=torch.float32)
            rois_t    = torch.tensor(valid_rois,    dtype=torch.float32)

            # ── Classification Loss ──
            cls_loss = cls_criterion(cls_scores, labels_t)

            # ── Bounding Box Regression Loss (chỉ tính cho positive) ──
            pos_mask = labels_t > 0
            bbox_loss = torch.tensor(0.0).to(device)
            if pos_mask.sum() > 0:
                pos_deltas = bbox_deltas[pos_mask]      # (Npos, 4*(C+1))
                pos_labels = labels_t[pos_mask]         # (Npos,)
                pos_gt     = gt_boxes_t[pos_mask]       # (Npos, 4)
                pos_rois   = rois_t[pos_mask]           # (Npos, 4)

                # Encode GT → delta targets
                targets = encode_bbox(pos_rois, pos_gt).to(device)  # (Npos, 4)

                # Lấy delta của đúng class được predict
                batch_idx = torch.arange(pos_labels.size(0))
                start = (pos_labels * 4)                # offset
                # Gather 4 giá trị (dx,dy,dw,dh) của class đúng
                pred_deltas = torch.stack([
                    pos_deltas[batch_idx, start + k] for k in range(4)
                ], dim=1)   # (Npos, 4)

                bbox_loss = bbox_criterion(pred_deltas, targets)

            # ── Multi-task Loss (đúng lý thuyết Fast RCNN) ──
            loss = cls_loss + bbox_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Metrics
            preds = cls_scores.argmax(dim=1)
            total_correct += (preds == labels_t).sum().item()
            total_samples += len(labels_t)
            total_cls_loss  += cls_loss.item()
            total_bbox_loss += bbox_loss.item()

            pbar.set_postfix({
                "cls_loss" : f"{total_cls_loss/(total_samples+1e-6):.4f}",
                "bbox_loss": f"{total_bbox_loss/(total_samples+1e-6):.4f}",
                "acc"      : f"{total_correct/max(total_samples,1):.3f}"
            })

        print(f"✅ Epoch {epoch+1} done\n")

    return model


# Chạy
model = FastRCNN(num_classes=NUM_CLASSES)
trained_model = train_fast_rcnn(model, DATA_ROOT, num_epochs=3, lr=1e-4)
torch.save(trained_model.state_dict(), "/content/drive/My Drive/RCNN/fast_rcnn.pth")
print("Đã lưu model!")


Training Fast RCNN trên 7324 ảnh | device=cpu


Epoch 1/3:   0%|          | 0/7324 [00:00<?, ?img/s]/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
Epoch 1/3:   7%|▋         | 539/7324 [47:27<9:57:20,  5.28s/img, cls_loss=0.0132, bbox_loss=0.0188, acc=0.793] 


KeyboardInterrupt: 